# Manual Price Push

Push manual price overrides for specific SKUs using market data or fixed prices.

## How to use
1. Run cells 1-3 (setup + data loading)
2. Edit the `PUSH_LIST` in cell 4 with your product IDs and desired actions
3. Run cell 4 — prices are computed per product × cohort (using cohort-specific market data)
4. Review the output table
5. Set `MODE = 'live'` and run cell 5 to push

> Each product is automatically expanded to all 9 cohorts. Market-based actions use cohort-specific prices.  
> Main/general cohorts (695, 61, 699, 697, 698, 696) are auto-mirrored by the push handler.

## Available Actions
| Action | Description |
|--------|-------------|
| `market_min` | Lowest market price (P0 / minimum) |
| `market_25` | 25th percentile market price |
| `market_50` | Median market price (P50) |
| `market_75` | 75th percentile market price |
| `market_max` | Highest market price (maximum) |
| `market_avg` | Average of min and max market prices |
| `target_margin` | Price from brand-category target margin |
| `step_up` | Next tier above current price in the price list |
| `step_down` | Next tier below current price in the price list |
| `<number>` | Fixed price (e.g. `115`, `23.5`) |

> Only SKUs with stock > 0 are pushed.

In [37]:
# =============================================================================
# SETUP
# =============================================================================
import sys, os
import pandas as pd
import numpy as np
import pytz
from datetime import datetime

sys.path.insert(0, os.path.abspath('.'))
import setup_environment_2
setup_environment_2.initialize_env()

from db import query_snowflake
from constants import WAREHOUSE_MAPPING, COHORT_IDS

os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')) or os.getcwd(), 'modules'))
%run push_prices_handler.ipynb
os.chdir('..')

CAIRO_TZ = pytz.timezone('Africa/Cairo')
TODAY = datetime.now(CAIRO_TZ).strftime('%Y-%m-%d')
print(f'Ready. Today: {TODAY}')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
Push Prices Handler loaded at 2026-04-16 16:47:42 Cairo time
✓ API credentials loaded successfully
✓ Google Sheets client initialized
Ready. Today: 2026-04-16


In [38]:
# =============================================================================
# LOAD MARKET DATA + WAC + CURRENT PRICES + PACKING UNITS
# =============================================================================
os.chdir('modules')
%run market_data_module_2.ipynb
os.chdir('..')

print('Loading market data (V2)...')
market_data = get_market_data_v2()
print(f'  Market data: {len(market_data)} rows')

print('Loading WAC...')
WAC_QUERY = f'''
SELECT product_id, wac_p
FROM finance.all_cogs c
WHERE wac_p > 0
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) 
        BETWEEN c.from_date AND c.to_date
'''
df_wac = query_snowflake(WAC_QUERY)
print(f'  WAC: {len(df_wac)} rows')

print('Loading current prices...')
PRICES_QUERY = f'''

    SELECT 
        cohort_id,
        pu.product_id,
        pu.packing_unit_id,
        pu.basic_unit_count,
        AVG(cpu.price) AS current_price
    FROM cohort_product_packing_units cpu
    JOIN packing_unit_products pu ON pu.id = cpu.product_packing_unit_id
    WHERE cpu.cohort_id IN ({','.join(str(c) for c in COHORT_IDS)})
        AND cpu.created_at::DATE <> '2023-07-31'
        AND cpu.is_customized = TRUE
        and basic_unit_count = 1 
    GROUP BY 1,2,3,4

'''
df_prices = query_snowflake(PRICES_QUERY)
#df_prices = setup_environment_2.dwh_pg_query(PRICES_QUERY, columns=['cohort_id','product_id','packing_unit_id','basic_unit_count','current_price'])
print(f'  Prices: {len(df_prices)} rows')

print('Loading packing units...')
PU_QUERY = '''
WITH sales_check AS (
    SELECT DISTINCT
        pso.product_id,
        pso.packing_unit_id,
        SUM(pso.total_price) AS nmv
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    WHERE so.created_at >= CURRENT_DATE - 60 
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY 1, 2
)
SELECT product_id, packing_unit_id, basic_unit_count
FROM (
    SELECT *, MAX(nmv) OVER (PARTITION BY product_id, is_basic_unit) AS top_nmv
    FROM (
        SELECT 
            pup.product_id,
            pup.packing_unit_id,
            pup.basic_unit_count,
            pup.is_basic_unit,
            COUNT(DISTINCT CASE WHEN pup.basic_unit_count = 1 THEN pup.packing_unit_id END) 
                OVER (PARTITION BY pup.product_id) AS total_basic,
            COALESCE(sc.nmv, 0) AS nmv
        FROM packing_unit_products pup
        LEFT JOIN sales_check sc 
            ON sc.product_id = pup.product_id 
            AND sc.packing_unit_id = pup.packing_unit_id
        WHERE pup.deleted_at IS NULL
    )
)
WHERE nmv = top_nmv OR (top_nmv = 0 AND total_basic <= 1)
'''
df_pus = query_snowflake(PU_QUERY)
print(f'  Packing units: {len(df_pus)} rows')

print('Loading product info...')
PRODUCT_QUERY = '''
SELECT p.id AS product_id,
       CONCAT(p.name_ar, ' ', p.size, ' ', product_units.name_ar) AS sku,
       b.name_ar AS brand,
       c.name_ar AS cat
FROM products p
JOIN brands b ON b.id = p.brand_id
JOIN categories c ON c.id = p.category_id
JOIN product_units ON product_units.id = p.unit_id
'''
df_products = query_snowflake(PRODUCT_QUERY)
print(f'  Products: {len(df_products)} rows')

print('Loading target margins...')
MARGIN_QUERY = f'''
SELECT brand, cat, target_margin
FROM (
    SELECT b.name_ar AS brand, c.name_ar AS cat, cplan.margin AS target_margin,
           ROW_NUMBER() OVER (PARTITION BY b.name_ar, c.name_ar ORDER BY cplan.date DESC) AS rn
    FROM performance.commercial_targets cplan
    JOIN brands b ON b.id = cplan.brand_id
    JOIN categories c ON c.id = cplan.category_id
    WHERE cplan.margin IS NOT NULL
    QUALIFY CASE 
        WHEN DATE_TRUNC('month', MAX(date) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) 
        THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
        ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month') 
    END = DATE_TRUNC('month', date)
)
WHERE rn = 1
'''
df_targets = query_snowflake(MARGIN_QUERY)
print(f'  Target margins: {len(df_targets)} rows')

print('Loading stocks...')
STOCKS_QUERY = '''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
)
SELECT warehouse_id, product_id,
    CASE WHEN stocks_child IS NOT NULL AND stocks = 0 THEN stocks_child ELSE stocks END AS stocks
FROM (
    SELECT pw.warehouse_id, pw.product_id,
        pw.available_stock::INTEGER AS stocks,
        pw2.available_stock::INTEGER AS stocks_child
    FROM product_warehouse pw
    LEFT JOIN parent_whs p ON p.parent_id = pw.warehouse_id
    LEFT JOIN product_warehouse pw2 ON pw2.warehouse_id = p.child_id AND pw.product_id = pw2.product_id
    WHERE pw.warehouse_id NOT IN (6, 9, 10)
        AND pw.is_basic_unit = 1
)
'''
df_stocks = query_snowflake(STOCKS_QUERY)
print(f'  Stocks: {len(df_stocks)} rows')

print('\nAll data loaded.')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
Market Data Module loaded at 2026-04-16 16:47:43 Cairo time
Snowflake timezone: America/Los_Angeles
All queries defined ✓
Helper functions defined ✓
get_market_data() function defined ✓
get_margin_tiers() function defined ✓
get_brand_market_percentiles() function defined ✓
fill_brand_market_fallback() function defined ✓

MARKET DATA MODULE READY

Available functions (NO INPUT REQUIRED):
  - get_market_data()              : Fetch and process all market prices
  - get_margin_tiers()             : Fetch and calculate margin tiers
  - get_brand_market_percentiles() : Fetch brand-level market margin percentiles
  - fill_brand_market_fallback()   : Fill NaN market cols with brand percentiles

Usage:
  %run market_data_module.ipynb
  df_market = get_market_data()
  df_tiers = get_margin_tiers()
  df_brand_percs = get_brand_market_percentiles()
  df = fill_brand_market_fallback(df, df_brand_percs)
get_market_signals() function de

In [39]:
# =============================================================================
# BUILD LOOKUP TABLE (product_id → market prices + WAC + target margin)
# =============================================================================

# V2 market_data returns (product_id, region, price_tiers, wac_p, target_margin, num_sources, market_data_source)
# Expand to cohorts for lookup
df_market_cohorts = expand_to_cohorts(market_data)

# Lookup is per (product_id, cohort_id)
lookup = df_products.merge(df_wac, on='product_id', how='left')

cohort_df = pd.DataFrame({'cohort_id': COHORT_IDS})
lookup = lookup.merge(cohort_df, how='cross')
lookup = lookup.merge(
    df_market_cohorts[['product_id', 'cohort_id', 'price_tiers', 'target_margin']],
    on=['product_id', 'cohort_id'], how='left'
)
lookup['price_tiers'] = lookup['price_tiers'].apply(lambda x: x if isinstance(x, list) else [])

# Fill target_margin for rows without V2 data
lookup = lookup.merge(df_targets, on=['brand', 'cat'], how='left', suffixes=('', '_tgt'))
for col in ['target_bm', 'target_bm_tgt', 'cat_target_margin', 'cat_target_margin_tgt']:
    if col in lookup.columns:
        lookup['target_margin'] = lookup['target_margin'].fillna(lookup[col])
lookup['target_margin'] = lookup['target_margin'].fillna(0.05)
lookup.drop(columns=[c for c in lookup.columns if c.startswith('target_bm') or c.startswith('cat_target')], inplace=True, errors='ignore')

# Derive market percentile prices from price_tiers list
def tiers_to_percentiles(tiers):
    if not tiers or len(tiers) == 0:
        return pd.Series({'market_min': np.nan, 'market_25': np.nan, 'market_50': np.nan,
                          'market_75': np.nan, 'market_max': np.nan, 'market_avg': np.nan})
    arr = np.array(tiers)
    return pd.Series({
        'market_min': arr[0],
        'market_25': np.percentile(arr, 25),
        'market_50': np.percentile(arr, 50),
        'market_75': np.percentile(arr, 75),
        'market_max': arr[-1],
        'market_avg': (arr[0] + arr[-1]) / 2,
    })

lookup = pd.concat([lookup, lookup['price_tiers'].apply(tiers_to_percentiles)], axis=1)

# Merge current base-unit price per (product_id, cohort_id)
base_prices = df_prices[df_prices['basic_unit_count'] == 1][['cohort_id', 'product_id', 'current_price']].drop_duplicates()
lookup = lookup.merge(base_prices, on=['product_id', 'cohort_id'], how='left')

# Merge stocks (sum across warehouses per product for display; warehouse-level for push filtering)
wh_df = pd.DataFrame(WAREHOUSE_MAPPING, columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])
product_stocks = df_stocks.groupby('product_id')['stocks'].sum().reset_index().rename(columns={'stocks': 'total_stocks'})
lookup = lookup.merge(product_stocks, on='product_id', how='left')
lookup['total_stocks'] = lookup['total_stocks'].fillna(0).astype(int)

print(f'Lookup table: {len(lookup)} rows (product x cohort)')
print(f'  With market data: {(lookup["price_tiers"].apply(len) > 0).sum()}')
print(f'  With WAC: {lookup["wac_p"].notna().sum()}')
print(f'  With target margin: {lookup["target_margin"].notna().sum()}')

Lookup table: 232277 rows (product x cohort)
  With market data: 44231
  With WAC: 75956
  With target margin: 232277


In [54]:
lookup[(lookup['product_id']== 22573)].price_tiers.values[0]

[360.0,
 375.5,
 379.25,
 380.0,
 384.75,
 389.5,
 394.5,
 399.5,
 405.25,
 411.0,
 417.0,
 423.25,
 429.75,
 434.75,
 439.75,
 445.0,
 451.5,
 458.0]

In [43]:
lookup[(lookup['current_price'] > lookup['market_max'])&(lookup['total_stocks']>0) ].to_excel('manual_data.xlsx')

In [44]:
# =============================================================================
# DEFINE YOUR SKU ACTIONS HERE
# =============================================================================
# Format: (product_id, action)
#   action can be: 'market_min', 'market_25', 'market_50', 'market_75',
#                  'market_max', 'market_avg', 'target_margin',
#                  'step_up', 'step_down', or a number (fixed price)
#
# step_up: next tier above current price in the price_tiers list
# step_down: next tier below current price in the price_tiers list
# Each product is automatically expanded to ALL cohorts.
# Market-based actions use the cohort-specific market price.
# Only SKUs with stock > 0 are pushed.

PUSH_LIST = [
(13325,704,23.75),
(13325,703,23.875),
(13325,700,24.25),
(13325,701,24.25),
(13325,1125,24.5),
(25710,703,42.25),
(13326,704,50.375),
(13326,703,50.625),
(13326,700,52),
(13326,701,51.75),
(12497,701,58.25),
(12493,704,50.5),
(12493,703,50.625),
(12493,701,51.75),
(13323,700,64.25),
(12493,700,52),
(12493,1126,52),
(25709,704,42.25),
(25710,704,42.25),
(25709,703,42.375),
(25712,701,54),
(25711,703,53.25),
(25711,704,53.25),
(25710,700,44),
(25710,701,44),
(25710,1125,44),
(25711,700,53.75),
(25711,1123,53.75),
(25711,1125,53.75),
(25712,700,53.75),
(25712,703,53.75),
(25712,704,53.75),
(25713,700,53.75),
(25713,703,53.75),
(25713,704,53.75),
(25711,701,54),
(25713,701,54),
(25712,1123,54.25),
(25713,1123,54.25),
(25713,1125,54.25),
(10397,700,92.75),
(10397,701,92.75),
(10397,1126,92.75),
(25709,700,46),
(25709,701,46),
(25709,1125,46),
(10397,704,93.5),
(12494,700,274.375),
(3028,700,197),
(3028,701,197),
(3028,702,197),
(3028,703,197),
(3028,704,197),
(3028,1125,197),
(3028,1126,197),
(12494,704,265.5),
(12494,703,267),
(12494,701,272.875),
(12494,1123,274),
(12494,1124,274),
(12494,1125,274),
(12494,1126,274),
(12494,702,274.375),
(1722,702,141.375),
(1722,703,142.75),
(1722,704,142.75),
(1722,700,144.125),
(1722,701,144.125),
(1722,1123,144.125),
(1722,1125,144.125),
(1722,1126,144.125),
(10722,704,222.25),
(10722,703,223.5),
(10722,701,228.625),
(10722,1123,229.5),
(10722,1125,229.5),
(10722,1126,229.5),
(10722,700,229.625),
(10722,702,229.625),
(12492,704,95.5),
(12492,703,96.125),
(12492,701,98.125),
(12492,1123,98.5),
(12492,1125,98.5),
(12492,1126,98.5),
(12492,700,98.75),
(12492,702,98.75),
(9816,702,71),
(9816,703,71),
(9816,704,71),
(9816,1123,71),
(9816,1124,71),
(9816,1125,71),
(9816,1126,71),
(9815,702,71),
(9815,703,71),
(9815,704,71),
(9815,1123,71),
(9815,1124,71),
(9815,1125,71),
(9815,1126,71),
(11327,1124,60.25),
(21686,1126,27),
(11164,701,71.25),
(11164,1123,71.25),
(11164,1124,71.25),
(11164,1126,71.25),
(11164,703,71.5),
(11164,704,71.5),
(11164,700,72.5),
(21793,700,164.125),
(21793,702,164.125),
(21793,1126,164.125),
(442,700,714.75),
(442,701,714.75),
(442,702,714.75),
(442,1123,714.75),
(442,1124,714.75),
(442,1125,714.75),
(442,1126,714.75),
(21793,701,166.5),
(21793,703,166.5),
(21793,704,169),
(11180,703,433.75),
(26128,700,63.5),
(23030,700,84),
(23030,1126,84),
(1059,704,115.25),
(23028,704,84.25),
(1059,703,117.5),
(12979,700,49),
(5853,700,212.75),
(5853,701,212.75),
(5853,702,212.75),
(5853,1123,212.75),
(5853,1125,212.75),
(13198,700,533.5),
(3080,700,205.25),
(3080,701,205.25),
(8356,704,398),
(23028,1123,85),
(5848,700,214),
(9250,703,52),
(3080,704,207.625),
(7007,1125,391.5),
(7007,1126,391.5),
(7007,703,394.5),
(8636,702,95),
(7005,1125,392),
(3079,703,208.5),
(3072,701,208.5),
(5850,1125,216.75),
(19997,704,477.75),
(19997,703,478.25),
(8897,702,105),
(8897,1125,105),
(3073,701,206.875),
(9998,701,260),
(8898,1124,105),
(26126,702,28),
(8356,1125,398),
(21081,700,92.75),
(20927,1125,399),
(11295,702,921.75),
(7007,704,398),
(942,704,101.25),
(19972,1124,99),
(19972,1125,99),
(19972,1126,99),
(3079,701,208.5),
(14039,1123,110.25),
(14039,1126,110.25),
(19971,1123,99.25),
(19971,1124,99.25),
(19971,1125,99.25),
(21798,704,231.25),
(22866,702,40.75),
(4594,701,221),
(332,703,415),
(8898,702,105),
(2193,703,297),
(2203,703,297),
(3338,703,375.125),
(4176,1123,78.5),
(12300,1123,204.5),
(12300,1124,204.5),
(8059,704,46.25),
(8058,704,46.25),
(2193,1123,297),
(2193,1124,297),
(2193,1126,297),
(6023,1125,789),
(737,1123,789),
(737,1124,789),
(737,1125,789),
(737,1126,789),
(737,704,775.5),
(5080,700,273.75),
(5080,701,273.75),
(2203,1124,298.375),
(2203,1126,298.375),
(22866,703,40.75),
(6023,701,788.5),
(9251,703,55.25),
(10466,1123,375),
(10466,1125,375),
(10466,1126,375),
(14038,701,112.5),
(2668,1126,641),
(6023,702,789),
(8908,701,362),
(8897,1123,105),
(6023,700,789),
(9396,704,168.75),
(12300,1125,204.5),
(10884,701,59.75),
(11936,700,224),
(13198,701,533.5),
(13198,1123,533.5),
(13198,1124,533.5),
(27215,700,111.875),
(27215,1126,111.875),
(27214,700,111.875),
(27216,700,111.875),
(27216,1126,111.875),
(9250,702,51.625),
(27212,700,279.875),
(4592,701,216.75),
(21798,703,230.75),
(2217,702,26.5),
(12598,1125,820),
(12598,1126,820),
(8908,1123,364.625),
(8360,703,376.375),
(4253,701,222.5),
(6030,1124,799),
(6030,1126,799),
(6029,1123,799),
(6029,1124,799),
(4967,1125,94.75),
(4967,1126,94.75),
(23278,700,125.25),
(8898,1123,105),
(8894,1125,799.5),
(8894,1126,799.5),
(4071,704,785.25),
(8156,700,348.75),
(5209,700,56.25),
(25718,1123,282.5),
(3913,700,91.25),
(2673,1126,648),
(5080,1123,273.75),
(5080,1124,273.75),
(5080,1125,273.75),
(5080,1126,273.75),
(9250,704,52),
(11887,703,170),
(24735,701,184.5),
(24735,702,184.5),
(8894,701,799),
(4071,702,799),
(8894,702,799.5),
(3557,1123,51),
(3557,1124,51),
(3557,1126,51),
(3049,701,215),
(2658,702,461.5),
(2661,1123,461.75),
(11812,701,134.5),
(1512,1125,51.5),
(11895,1125,461.75),
(11895,1126,461.75),
(4253,700,222.5),
(4253,702,222.5),
(4253,1123,222.5),
(4253,1124,222.5),
(4253,1125,222.5),
(4253,1126,222.5),
(12595,1126,527.75),
(9379,700,97),
(21700,702,93.5),
(12674,703,175.25),
(9430,702,427.5),
(9430,1125,427.5),
(9430,1126,427.5),
(11625,700,361.25),
(9430,700,427.5),
(9430,701,427.5),
(11812,700,134.5),
(11812,702,134.5),
(11812,1124,134.5),
(25539,700,185.875),
(20629,702,185.875),
(2196,702,336.75),
(11678,700,85.25),
(11678,701,85.25),
(11678,1123,85.25),
(11678,1124,85.25),
(11678,1125,85.25),
(10580,702,157.75),
(10580,1125,157.75),
(24567,700,104.25),
(24567,702,104.25),
(24567,1123,104.25),
(24567,1124,104.25),
(24567,1125,104.25),
(24567,1126,104.25),
(10721,702,323.25),
(11409,704,58),
(6966,704,109),
(3047,703,109),
(3047,704,109),
(9998,700,260),
(6344,701,110.625),
(9430,704,431.375),
(23238,703,472.25),
(10170,700,623.75),
(10170,701,623.75),
(10170,703,623.75),
(10170,704,623.75),
(10170,1124,623.75),
(10170,1125,623.75),
(10170,1126,623.75),
(2100,702,351.5),
(23237,703,443),
(12674,700,177),
(12674,702,177),
(12674,1123,177),
(12674,1124,177),
(12674,1125,177),
(12674,1126,177),
(20602,700,187.25),
(20602,701,187.25),
(20602,702,187.25),
(20627,701,187.25),
(20627,702,187.25),
(11868,1123,108),
(10581,1125,160),
(3557,704,51),
(19931,1124,81.125),
(7007,701,391),
(22570,701,142),
(9396,700,166.75),
(11183,700,86.5),
(10455,703,472),
(21705,704,41.25),
(11413,703,478),
(10455,704,472),
(12295,702,203),
(11642,704,367),
(11399,1125,646.5),
(3047,700,109.375),
(3047,701,109.375),
(3047,702,109.375),
(1433,701,271.125),
(12638,700,142.25),
(13198,702,531.75),
(9396,701,168.25),
(12538,700,77),
(380,1123,516.75),
(380,1124,516.75),
(11771,704,143.75),
(1512,700,51.5),
(1512,701,51.5),
(1512,704,51.5),
(1512,1124,51.5),
(10529,701,393.875),
(6966,700,109.75),
(6966,701,109.75),
(22569,1123,377),
(22569,1125,377),
(22569,1126,377),
(1163,1126,107.75),
(12674,704,175.25),
(11183,701,87),
(11183,702,87),
(11183,1124,87),
(11183,1126,87),
(11771,703,143.75),
(9459,704,715.5),
(22327,704,782),
(22570,700,142.75),
(22570,702,142.75),
(11161,703,61.75),
(11160,703,61.75),
(3387,1123,272.5),
(3387,1125,272.5),
(3388,1123,272.5),
(3388,1125,272.5),
(11183,703,87.25),
(12513,702,598.375),
(12513,701,598.625),
(1511,700,51.75),
(1511,701,51.75),
(1511,702,51.75),
(1511,703,51.75),
(1511,1126,51.75),
(3394,1124,225),
(5128,1123,270.25),
(5128,1124,270.25),
(5128,1125,270.25),
(523,701,205),
(22361,1123,784.5),
(22361,1125,784.5),
(12513,700,598.375),
(22866,1123,40.75),
(22363,1126,681.5),
(5604,700,870.25),
(5208,1125,55.625),
(24247,701,100),
(22570,703,141.5),
(19931,1123,81.125),
(19931,1126,81.125),
(24758,700,327.5),
(22884,704,50.5),
(23690,701,158.5),
(1513,700,51.875),
(1513,701,51.875),
(1513,702,51.875),
(1513,703,51.875),
(1207,704,34.75),
(11771,700,143.75),
(11771,701,143.75),
(9998,1123,260),
(9998,1125,260),
(12638,1123,144.25),
(22570,1124,144.25),
(4253,703,214.5),
(4253,704,214.5),
(12513,1123,603.5),
(12513,1126,603.5),
(5604,1125,877.5),
(5604,1126,877.5),
(5603,1123,877.5),
(5603,1125,877.5),
(5603,1126,877.5),
(5604,702,873.5),
(11897,1126,358.5),
(5604,701,874.125),
(11771,702,143.75),
(11771,1123,143.75),
(11771,1124,143.75),
(11771,1125,143.75),
(11771,1126,143.75),
(11885,701,170),
(11887,701,170),
(22570,704,142),
(5208,700,55.625),
(21468,701,40.25),
(19932,700,81),
(19932,702,81),
(19931,700,81),
(19931,702,81),
(5604,704,863.25),
(5453,700,299),
(471,701,727.5),
(12638,701,144.25),
(19685,700,930.5),
(12638,704,142.25),
(19932,701,81.125),
(24195,701,43.75),
(5603,702,876.75),
(24384,702,389.75),
(5603,701,877.5),
(3399,703,550),
(12295,700,203),
(12295,703,203),
(12295,704,203),
(5603,700,876.75),
(24384,700,389.75),
(22572,700,389.75),
(19931,701,81.25),
(20984,701,791.25),
(20984,1123,791.25),
(20984,1124,791.25),
(20984,1125,791.25),
(9459,703,723.25),
(10720,1123,340.625),
(10720,1125,340.625),
(6965,702,861.75),
(6965,1124,861.75),
(3388,700,274.625),
(22971,1125,1038.75),
(22971,1126,1038.75),
(12508,1124,343.75),
(22572,1123,392.875),
(22572,1124,392.875),
(22572,1125,392.875),
(12508,1126,343.75),
(3395,700,420),
(10720,703,340.625),
(22866,704,40.75),
(24384,704,386.25),
(12508,704,338),
(3394,704,220.5),
(3399,704,551.75),
(9493,704,152.25),
(13270,704,318.5),
(9459,700,721.25),
(11161,700,62.75),
(11160,700,62.75),
(1085,1123,960.5),
(1085,1124,960.5),
(1085,1125,960.5),
(1085,1126,960.5),
(9459,1125,720.75),
(9459,1126,720.75),
(24384,701,392.75),
(22572,701,392.75),
(11885,700,170.875),
(11887,700,170.875),
(9459,701,723.25),
(11092,700,399),
(22572,703,388),
(3132,701,54.25),
(12536,700,894.25),
(22088,703,39.875),
(22088,704,39.875),
(5644,702,976.5),
(380,701,520.5),
(11339,704,111.5),
(947,703,64.5),
(9963,1125,50.5),
(12665,1123,282.25),
(12665,1125,282.25),
(12665,1126,282.25),
(410,702,291),
(12538,703,77),
(5208,1124,55.625),
(22971,704,1031.25),
(22088,700,40),
(22088,701,40),
(22088,1123,40),
(22088,1124,40),
(22088,1125,40),
(11412,703,486.75),
(26594,704,420.25),
(11412,704,486.75),
(22971,703,1032.625),
(5080,703,273.75),
(5642,702,976.5),
(11160,704,62),
(11091,704,400.875),
(169,701,139.25),
(1513,704,51.875),
(20629,703,190),
(24124,1126,1214.75),
(9493,702,151.75),
(9182,700,224.125),
(9182,701,224.125),
(2188,1124,39.25),
(21171,700,60.5),
(24668,701,1079.75),
(13270,700,318.75),
(13270,701,318.75),
(13270,1123,318.75),
(13270,1124,318.75),
(13270,1125,318.75),
(23991,703,388.75),
(23992,702,388.75),
(23992,703,388.75),
(23992,1125,388.75),
(10524,700,388.75),
(10524,701,388.75),
(10524,703,388.75),
(10524,1123,388.75),
(6266,1124,49),
(4076,701,147),
(4966,1125,313.5),
(5080,702,273.75),
(13247,701,391.25),
(13247,702,391.25),
(344,701,386.5),
(1051,1126,653),
(1052,1126,653),
(13259,704,458.5),
(1514,1123,253.75),
(1514,1125,253.75),
(1514,1126,253.75),
(13270,702,320.375),
(26594,702,418.875),
(13271,702,419.625),
(13276,1124,415.25),
(2505,1124,96),
(12664,1123,285),
(12664,1124,285),
(12664,1125,285),
(12664,1126,285),
(13276,700,415.25),
(13276,701,415.25),
(20629,704,190),
(10605,700,553.375),
(10603,700,553.375),
(10604,700,553.375),
(10603,703,557.25),
(10604,703,557.25),
(9251,702,55.25),
(3986,701,81.5),
(3986,1123,81.5),
(3986,1125,81.5),
(3048,701,107.75),
(10605,1123,554.25),
(10605,1125,554.25),
(10603,1123,554.25),
(10603,1124,554.25),
(10603,1125,554.25),
(10604,1123,554.25),
(10604,1124,554.25),
(10604,1126,554.25),
(11427,704,67.375),
(6968,1124,880),
(13247,1123,392.75),
(13247,1124,392.75),
(13247,1126,392.75),
(22680,701,758.375),
(22680,702,758.375),
(22680,1123,758.375),
(22680,1124,758.375),
(22680,1125,758.375),
(13247,700,392.25),
(13271,1125,421.25),
(20602,703,192),
(10603,704,555.25),
(10604,704,555.25),
(26594,1124,420.75),
(26594,1126,420.75),
(13271,701,421.25),
(6266,700,48.5),
(8637,1126,93.75),
(21059,1124,97.25),
(10605,701,558.5),
(10603,701,558.5),
(10604,701,558.5),
(1604,701,140),
(26390,700,49.25),
(4068,700,161.125),
(4068,701,161.125),
(4068,702,161.125),
(4068,704,161.125),
(4068,1123,161.125),
(4068,1124,161.125),
(4068,1125,161.125),
(24889,700,170),
(5831,1124,964.5),
(5831,1125,964.5),
(2406,700,154.75),
(2406,701,154.75),
(2406,702,154.75),
(2406,703,154.75),
(2406,704,154.75),
(2406,1123,154.75),
(2406,1124,154.75),
(2406,1125,154.75),
(2406,1126,154.75),
(1514,700,256.25),
(1514,701,256.25),
(1514,702,256.25),
(1514,703,256.25),
(13213,701,178.75),
(20627,703,190),
(10530,700,424.5),
(10530,701,424.5),
(10530,702,424.5),
(10530,703,424.5),
(10530,704,424.5),
(10530,1125,424.5),
(10530,1126,424.5),
(10411,1123,42.375),
(22923,701,102.577309236948),
(10527,1125,33.5),
(13030,703,199),
(13030,1125,199),
(13030,1126,199),
(10462,701,49.25),
(10462,1123,49.25),
(10462,1124,49.25),
(10462,1125,49.25),
(10462,1126,49.25),
(25342,703,1225),
(12550,1125,518.25),
(12295,1124,210),
(332,700,423.772545602132),
(823,1124,945.625),
(10534,1126,186.5),
(22436,700,35),
(8682,701,92.375),
(5645,703,972.125),
(5644,703,972.125),
(5642,703,972.125),
(3549,700,155.75),
(3549,701,155.75),
(3549,702,155.75),
(3549,703,155.75),
(3549,704,155.75),
(3549,1123,155.75),
(3549,1124,155.75),
(3549,1125,155.75),
(3549,1126,155.75),
(8050,703,215.75),
(8050,1123,215.75),
(8050,1125,215.75),
(13190,1124,206.125),
(354,700,423.783833648909),
(11421,1123,177.5),
(11421,1124,177.5),
(11421,1125,177.5),
(11421,1126,177.5),
(5406,703,215.875),
(5406,704,215.875),
(10408,1123,42.625),
(25342,1126,1228.75),
(25287,700,24.5),
(8157,700,344),
(8157,701,344),
(8157,702,344),
(8157,703,344),
(8157,1125,344),
(9085,700,344),
(9085,701,344),
(9085,702,344),
(9085,703,344),
(23994,700,427.5),
(23994,701,427.5),
(23994,702,427.5),
(23994,703,427.5),
(23994,704,427.5),
(23994,1125,427.5),
(24194,700,45.25),
(26389,700,50),
(23276,701,25),
(12674,701,177),
(5644,1126,974.75),
(5642,1124,974.75),
(8990,702,296.75),
(8990,1125,296.75),
(13191,1123,207),
(13191,1124,207),
(13191,1125,207),
(13192,703,207.125),
(10670,1123,86.8347248696755),
(10670,1126,86.8347248696755),
(2709,701,68.875),
(2709,1123,68.875),
(2709,1124,68.875),
(9430,703,406.5),
(10460,1123,49.25),
(10460,1124,49.25),
(10460,1125,49.25),
(10460,1126,49.25),
(563,700,423.908431418664),
(9087,700,298.25),
(9087,701,298.25),
(9087,703,298.25),
(9087,1123,298.25),
(9087,1125,298.25),
(9087,1126,298.25),
(2418,703,298.25),
(2709,700,68.875),
(22680,703,757.125),
(10527,1123,33.5),
(13193,703,208.25),
(13193,1125,208.25),
(22876,703,122.25),
(10461,1123,49.25),
(10461,1124,49.25),
(10461,1125,49.25),
(10461,1126,49.25),
(22923,700,102.577309236948),
(3855,1125,308.25),
(11856,701,306.38556835665),
(25991,700,145.5),
(25991,1125,145.5),
(10939,700,235.75),
(2709,703,68.875),
(12142,1125,460),
(3863,1123,316.5),
(3863,1126,316.5),
(22908,1125,49.375),
(22908,1126,49.375),
(1664,700,269.25),
(23491,701,144),
(9877,703,69.125),
(25629,700,49.75),
(13025,700,25),
(13025,701,25),
(13025,704,25),
(13025,1123,25),
(13025,1124,25),
(13025,1125,25),
(13025,1126,25),
(23276,703,25),
(23276,704,25),
(23276,1123,25),
(23276,1124,25),
(23276,1125,25),
(13024,704,25),
(13024,1123,25),
(6929,700,1783.125),
(1664,703,265.75),
(22680,704,760.75),
(2833,702,112.125),
(2505,1125,96),
(12641,703,468.5),
(10384,702,168.875),
(5853,703,202.25),
(5853,704,202.25),
(25539,704,197.75),
(1207,700,35.5),
(26447,702,342.75),
(26447,1123,342.75),
(26447,1124,342.75),
(26447,1125,342.75),
(26447,1126,342.75),
(10937,700,229.25),
(1666,704,267.875),
(1664,704,268.625),
(436,702,311.872620097203),
(21478,1123,40),
(10527,702,33.5),
(10527,1124,33.5),
(10668,703,88.5),
(3024,703,117),
(10586,703,108.5),
(3863,701,316.5),
(5848,703,204.25),
(5848,704,204.25),
(10384,1123,170.5),
(10384,1125,170.5),
(10384,1126,170.5),
(25636,1123,50.5),
(2833,700,113.5),
(2833,701,113.5),
(2833,1123,113.5),
(2833,1125,113.5),
(2833,1126,113.5),
(25549,701,85.25),
(3012,704,779.25),
(474,1123,501.625),
(474,1124,501.625),
(474,1125,501.625),
(24195,703,43.75),
(24195,1123,43.75),
(24195,1124,43.75),
(24195,1125,43.75),
(24195,1126,43.75),
(24194,703,44.5),
(6802,1124,45),
(3863,700,316.5),
(3012,703,779.25),
(8637,700,93.75),
(3542,700,804.75),
(3542,701,804.75),
(3542,1123,804.75),
(3542,1124,804.75),
(3542,1125,804.75),
(3542,1126,804.75),
(25287,702,24.5),
(25287,1123,24.5),
(25287,1124,24.5),
(25287,1125,24.5),
(25287,1126,24.5),
(23532,701,49.75),
(23532,702,49.75),
(23532,1123,49.75),
(2505,701,96),
(24195,702,44),
(24195,704,44),
(24194,702,44.75),
(24194,704,44.75),
(24194,1123,44.75),
(24194,1124,44.75),
(24194,1125,44.75),
(24194,1126,44.75),
(10884,700,59.75),
(10884,1123,59.75),
(10884,1124,59.75),
(13884,701,135),
(22914,701,107),
(22943,700,109.25),
(10940,700,235.5),
(12302,1126,174.75),
(10938,700,230.5),
(20969,703,50.5),
(2504,1123,52.5),
(2504,1124,52.5),
(9996,1124,247),
(9996,1125,247),
(1207,701,35.5),
(3012,1125,779.25),
(3012,1126,779.25),
(11791,700,539.75),
(11791,701,539.75),
(11791,702,539.75),
(11791,703,539.75),
(11791,1125,539.75),
(2486,701,102.75),
(9996,701,247),
(11161,704,62),
(10278,701,110.75),
(11164,702,71.25),
(2695,1124,49.25),
(2695,1125,49.25),
(2695,1126,49.25),
(22944,700,50),
(23533,701,50.25),
(10465,1123,408.5),
(10465,1124,408.5),
(10465,1125,408.5),
(10465,1126,408.5),
(7354,1123,128.5),
(7354,1125,128.5),
(7354,1126,128.5),
(586,701,525),
(586,702,525),
(586,704,525),
(22924,701,105),
(12586,701,52.75),
(22925,700,105.5),
(5579,1123,350.25),
(5579,1125,350.25),
(5579,1126,350.25),
(1664,702,272.75),
(10666,701,31.25),
(26755,701,735),
(12501,701,102),
(23377,700,35.5),
(23377,701,35.5),
(948,702,37.5),
(2695,701,49.25),
(5733,700,645)
 
]


# =============================================================================
# COMPUTE NEW PRICES (per product × cohort)
# =============================================================================
ACTION_TO_COLUMN = {
    'market_min': 'market_min',
    'market_25': 'market_25',
    'market_50': 'market_50',
    'market_75': 'market_75',
    'market_max': 'market_max',
    'market_avg': 'market_avg',
}

results = []
for product_id,cohort_id, action in PUSH_LIST:
    product_rows = lookup[(lookup['product_id'] == product_id)&(lookup['cohort_id'] == cohort_id)] #
    if product_rows.empty:
        results.append({'product_id': product_id, 'cohort_id': '-', 'action': str(action), 'status': 'NOT FOUND'})
        continue

    for _, row in product_rows.iterrows():
        cohort_id = int(row['cohort_id'])
        wac = row.get('wac_p', None)
        sku = row.get('sku', '')
        brand = row.get('brand', '')

        if isinstance(action, (int, float)):
            base_price = float(action)
            action_label = f'fixed_{action}'
        elif action == 'target_margin':
            tm = row.get('target_margin', 0.05)
            if pd.isna(wac) or wac <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO WAC'})
                continue
            base_price = wac / (1 - tm)
            action_label = f'target_margin ({tm:.1%})'
        elif action == 'step_up':
            tiers = row.get('price_tiers', [])
            cur = row.get('current_price', 0)
            if not tiers or pd.isna(cur) or cur <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO TIERS OR PRICE'})
                continue
            next_up = [t for t in tiers if t > cur + 0.25]
            if not next_up:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'ALREADY AT TOP'})
                continue
            base_price = next_up[0]
            action_label = f'step_up ({cur:.2f} -> {base_price:.2f})'
        elif action == 'step_down':
            tiers = row.get('price_tiers', [])
            cur = row.get('current_price', 0)
            if not tiers or pd.isna(cur) or cur <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO TIERS OR PRICE'})
                continue
            next_down = [t for t in reversed(tiers) if t < cur - 0.5]
            if not next_down:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'ALREADY AT BOTTOM'})
                continue
            base_price = next_down[0]
            action_label = f'step_down ({cur:.2f} -> {base_price:.2f})'
        elif action in ACTION_TO_COLUMN:
            col = ACTION_TO_COLUMN[action]
            val = row.get(col, None)
            if pd.isna(val):
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO MARKET DATA'})
                continue
            base_price = val
            action_label = action
        else:
            results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': str(action), 'status': 'INVALID ACTION'})
            continue

        base_price_rounded = np.round(base_price * 4) / 4
        margin = (base_price_rounded - wac) / base_price_rounded if (wac and wac > 0 and base_price_rounded > 0) else None

        cur_price = row.get('current_price', None)

        results.append({
            'product_id': product_id,
            'cohort_id': cohort_id,
            'sku': sku,
            'brand': brand,
            'action': action_label,
            'wac': wac,
            'current_price': cur_price,
            'base_price': base_price,
            'new_price': base_price_rounded,
            'margin': margin,
            'status': 'OK',
        })

df_results = pd.DataFrame(results)

ok_count = (df_results['status'] == 'OK').sum()
err_count = len(df_results) - ok_count
products_ok = df_results[df_results['status'] == 'OK']['product_id'].nunique()
print(f'Results: {ok_count} OK across {products_ok} products × cohorts, {err_count} errors/skipped')
if err_count > 0:
    print('\nErrors/Skipped:')
    print(df_results[df_results['status'] != 'OK'][['product_id', 'cohort_id', 'sku', 'action', 'status']].to_string(index=False))
df_results[df_results['status'] == 'OK']

Results: 930 OK across 341 products × cohorts, 0 errors/skipped


,product_id,cohort_id,sku,brand,action,wac,current_price,base_price,new_price,margin,status
0,13325,704,حلو الشام بيكنج بودر جديد - 16 جم,حلو الشام,fixed_23.75,20.604072,445.00,23.750,23.75,0.132460,OK
1,13325,703,حلو الشام بيكنج بودر جديد - 16 جم,حلو الشام,fixed_23.875,20.604072,445.00,23.875,24.00,0.141497,OK
2,13325,700,حلو الشام بيكنج بودر جديد - 16 جم,حلو الشام,fixed_24.25,20.604072,445.00,24.250,24.25,0.150348,OK
3,13325,701,حلو الشام بيكنج بودر جديد - 16 جم,حلو الشام,fixed_24.25,20.604072,445.00,24.250,24.25,0.150348,OK
4,13325,1125,حلو الشام بيكنج بودر جديد - 16 جم,حلو الشام,fixed_24.5,20.604072,445.00,24.500,24.50,0.159017,OK
...,...,...,...,...,...,...,...,...,...,...,...
925,23377,700,بسكويت بيمبو شوكو ماكس إكس لارج - 7 جنية,بيمبو,fixed_35.5,33.250000,35.75,35.500,35.50,0.063380,OK
926,23377,701,بسكويت بيمبو شوكو ماكس إكس لارج - 7 جنية,بيمبو,fixed_35.5,33.250000,35.75,35.500,35.50,0.063380,OK
927,948,702,حلواني حلاوة طحينية سادة - 280 جم,حلواني,fixed_37.5,33.947315,37.75,37.500,37.50,0.094738,OK
928,2695,701,الشمعدان ميني تيبو بشوكولاتة - 6 قطعه,الشمعدان,fixed_49.25,46.075000,49.50,49.250,49.25,0.064467,OK


In [45]:
df_results=df_results[df_results['current_price']>df_results['new_price']]
df_results['change'] = (df_results['new_price'] - df_results['current_price'])/df_results['current_price']
print(df_results['margin'].mean())
print(df_results['margin'].median())
print(df_results['margin'].min())
print(df_results['margin'].max())
df_results=df_results[~df_results['brand'].isin(['كوكا كولا','شويبس'])]
df_results

0.11539460076761766
0.11443917776190465
0.016674157303370886
0.30043212577973993


,product_id,cohort_id,sku,brand,action,wac,current_price,base_price,new_price,margin,status,change
0,13325,704,حلو الشام بيكنج بودر جديد - 16 جم,حلو الشام,fixed_23.75,20.604072,445.00,23.750,23.75,0.132460,OK,-0.946629
1,13325,703,حلو الشام بيكنج بودر جديد - 16 جم,حلو الشام,fixed_23.875,20.604072,445.00,23.875,24.00,0.141497,OK,-0.946067
2,13325,700,حلو الشام بيكنج بودر جديد - 16 جم,حلو الشام,fixed_24.25,20.604072,445.00,24.250,24.25,0.150348,OK,-0.945506
3,13325,701,حلو الشام بيكنج بودر جديد - 16 جم,حلو الشام,fixed_24.25,20.604072,445.00,24.250,24.25,0.150348,OK,-0.945506
4,13325,1125,حلو الشام بيكنج بودر جديد - 16 جم,حلو الشام,fixed_24.5,20.604072,445.00,24.500,24.50,0.159017,OK,-0.944944
...,...,...,...,...,...,...,...,...,...,...,...,...
925,23377,700,بسكويت بيمبو شوكو ماكس إكس لارج - 7 جنية,بيمبو,fixed_35.5,33.250000,35.75,35.500,35.50,0.063380,OK,-0.006993
926,23377,701,بسكويت بيمبو شوكو ماكس إكس لارج - 7 جنية,بيمبو,fixed_35.5,33.250000,35.75,35.500,35.50,0.063380,OK,-0.006993
927,948,702,حلواني حلاوة طحينية سادة - 280 جم,حلواني,fixed_37.5,33.947315,37.75,37.500,37.50,0.094738,OK,-0.006623
928,2695,701,الشمعدان ميني تيبو بشوكولاتة - 6 قطعه,الشمعدان,fixed_49.25,46.075000,49.50,49.250,49.25,0.064467,OK,-0.005051


In [34]:
# out = pd.read_excel('cohort-pricing-template.xlsx')
# id_ = 1255
# name = 'Mona_cairo'
# file_name_ = 'uploads/1_new_{}.xlsx'.format(name).replace(' ','_')
# out.to_excel(file_name_,index = False,engine = 'xlsxwriter')
# time.sleep(3)
# ################### Loop to avoid file limit ######################
# # split file into chunks
# print('Spliting file into chunks...')
# if id_ == 61:
#     chunks = [out[i:i + 2000] for i in range(0, len(out), 2000)]
# else:
#     chunks = [out[i:i + 4000] for i in range(0, len(out), 4000)]
# print(f'len chunks = {len(chunks)}')
# fileslist = []
# for i, chunk in tqdm(enumerate(chunks), total=len(chunks)):
#     fileslist.append(f'manual/output_{id_}_chunk_{i + 1}.xlsx')
#     output_file_path = f'manual/output_{id_}_chunk_{i + 1}.xlsx'
#     chunk.to_excel(output_file_path, index=False, engine='xlsxwriter')
# # Loop over chunks and upload
# print('Uploading...')
# for file in fileslist:
#     chunk = file.split('chunk_')[1].split('.xls')[0]
#     x = post_prices(id_, file)
#     # print(str(x.content))
#     if ('"success":true' in str(x.content).lower()):
#         print(f"Prices are upoladed successfuly Region: {name} ,cohort: {id_}, chunk: {chunk}")
#     else:
#         print(f"ERROR Region: {name} ,cohort: {id_}, chunk: {chunk}")
#         print(x.content)
#         final_status = False
#         break

In [46]:
# =============================================================================
# PUSH PRICES
# =============================================================================
MODE = 'live'  # Change to 'live' to actually push

df_ok = df_results[df_results['status'] == 'OK'].copy()

# Filter to only SKUs with stock
if len(df_ok) > 0:
    stock_by_product = df_stocks.groupby('product_id')['stocks'].sum().reset_index()
    df_ok = df_ok.merge(stock_by_product, on='product_id', how='left')
    no_stock = df_ok['stocks'].fillna(0) <= 0
    if no_stock.any():
        print(f'Skipping {no_stock.sum()} rows with no stock')
        df_ok = df_ok[~no_stock].copy()

if len(df_ok) == 0:
    print('No valid prices to push.')
else:
    wh_df = pd.DataFrame(WAREHOUSE_MAPPING, columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])

    push_rows = []
    for _, r in df_ok.iterrows():
        target_cohort = int(r['cohort_id'])
        cohort_whs = wh_df[wh_df['cohort_id'] == target_cohort]

        if cohort_whs.empty:
            cohort_whs = pd.DataFrame([{'warehouse_id': 1, 'cohort_id': target_cohort}])

        cur_price = r['current_price'] if pd.notna(r.get('current_price')) else 0

        for _, wh in cohort_whs.iterrows():
            push_rows.append({
                'product_id': int(r['product_id']),
                'sku': r['sku'],
                'new_price': r['new_price'],
                'warehouse_id': int(wh['warehouse_id']),
                'cohort_id': target_cohort,
                'stocks': 1,
                'current_price': cur_price,
            })

    df_push = pd.DataFrame(push_rows)

    n_products = df_ok['product_id'].nunique()
    n_cohorts = df_ok['cohort_id'].nunique()
    print(f'Pushing {n_products} products × {n_cohorts} cohorts = {len(df_push)} rows')
    print(f'Mode: {MODE}')
    print()

    result = push_prices(
        df_prices=df_push,
        pus=df_pus,
        source_module='manual_push',
        mode=MODE,
    )
    print(f'\nDone. Result: {result}')

Pushing 341 products × 9 cohorts = 1259 rows
Mode: live


🚀 MODE: LIVE
   Files will be prepared AND uploaded to API
Loading disable_pu_visibility from Google Sheets...
  ✓ Loaded 88 products to disable min PU visibility

PUSH PRICES - Source: manual_push
Total received: 1259
Price changes to push: 1259
Skipped (no change): 0

Price changes summary:
  Increases: 0
  Decreases: 1259

🔗 Mirrored prices to 6 main/general cohorts (+999 rows)
   Cohort 695 ← 700: 204 rows
   Cohort 61 ← 700: 204 rows
   Cohort 699 ← 702: 131 rows
   Cohort 697 ← 703: 163 rows
   Cohort 698 ← 704: 145 rows
   Cohort 696 ← 1123: 152 rows

📋 Prepared 2434 packing unit prices (incl. main cohorts)

Processing cohort: 61
  Saved: uploads/manual_push_61.xlsx (204 rows)
  Split into 1 chunks (size: 2000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 58.60it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

Processing cohort: 695
  Saved: uploads/manual_push_695.xlsx (204 rows)
  Split into 1 chunks (size: 4000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 58.43it/s]

...

  Uploading...


    ✓ Chunk 1 uploaded successfully

Processing cohort: 1126
  Saved: uploads/manual_push_1126.xlsx (134 rows)
  Split into 1 chunks (size: 4000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 78.96it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

🚀 UPLOAD COMPLETE
Mode: live
Total prepared: 2434
Total failed: 0
  Cleanup: removed 30 temporary .xlsx files from 2 folder(s)

Done. Result: {'total_received': 1259, 'price_changes': 1259, 'pushed': 2434, 'failed': 0, 'skipped': 0, 'source_module': 'manual_push', 'timestamp': '2026-04-16 17:05:27', 'mode': 'live', 'skipped_cohorts': []}
